# Causal Search: Migration from R (Kumu) to Python (PyTetrad)

This notebook demonstrates how to perform **BOSS (Best Order Score Search)** causal discovery using PyTetrad, replicating the functionality from Kumu's R implementation.

**Note**: This notebook uses **BOSS** as the primary algorithm, but also includes the **FGES** algorithm implementation as commented code for reference and easy switching between algorithms.

## What is BOSS?

BOSS is a **causal discovery algorithm** that learns causal relationships from observational data. It's not a "search" in the Google sense - rather, it's a statistical learning function that:
- Takes a **table of data** as input
- Performs **statistical analysis** to discover causal relationships using permutation-based approaches
- Outputs a **causal graph** showing which variables cause changes in other variables

BOSS works by building DAGs (Directed Acyclic Graphs) from permutations and has been tested to scale to 1000+ variables with excellent precision and recall.

## What is FGES? (Alternative Algorithm - Included as Commented Code)

FGES (Fast Greedy Equivalence Search) is another causal discovery algorithm that uses a greedy search strategy to find causal relationships. The FGES implementation is included throughout this notebook as commented code, allowing you to easily switch between BOSS and FGES.

## Three Key Components We'll Implement

1. **Score Configuration (SEM-BIC)**: How we evaluate the quality of potential causal graphs
2. **Bootstrapping**: How we ensure our results are robust by resampling (with algorithm-specific parameters)
3. **BOSS Algorithm** (with FGES alternative): The actual causal discovery process with its parameters

---

## 1. Install and Import Required Libraries

First, we need to install PyTetrad and import necessary libraries.

In [40]:
# Install PyTetrad (run this once)
# !pip install git+https://github.com/cmu-phil/py-tetrad

import pandas as pd
import numpy as np
import sys
import os
import io
import contextlib

# TetradSearch: Main interface wrapping Tetrad's Java functionality via JPype
import pytetrad.tools.TetradSearch as ts

### Understanding the Import

**`pytetrad.tools.TetradSearch`** is the key class that:
- Uses **JPype** to start a Java Virtual Machine (JVM)
- Loads the Tetrad JAR file containing all causal discovery algorithms
- Provides a Python-friendly interface to Java methods
- Handles data conversion between Pandas DataFrames and Tetrad's Java data structures

Unlike Kumu (R), which calls Tetrad-CMD as an external process via `system2()`, PyTetrad runs Tetrad **directly in-memory**, which is more efficient.

---

## 2. Load and Explore the Dataset

We'll use the `null_variable_dt.csv` file from Kumu's analysis.

In [41]:
data_path = "null_variable_dt.csv"
data = pd.read_csv(data_path)

print(f"Dataset shape: {data.shape[0]} rows × {data.shape[1]} columns")
print(f"First 5 columns: {data.columns.tolist()[:5]}")
data.head(3)

Dataset shape: 4870 rows × 138 columns
First 5 columns: ['start', 'mis_link', 'silence', 'code_dev', 'file']


,start,mis_link,silence,code_dev,file,mail_dev,thread,commit,churn,mis_link2,...,nv-mail_dev2,nv-thread2,nv-commit2,nv-churn2,nv-b_100433,nv-b_100740,nv-b_100742,nv-b_102939,nv-b_103864,nv-b_104180
0,1015182575,0.162007,1.452335,-0.834315,0.180391,0.889334,0.485497,0.530306,0.306018,0.501028,...,-1.054967,-0.568054,-0.170652,0.456812,0,0,0,0,0,0
1,1022958575,0.537005,1.841985,0.200191,0.842453,2.012817,1.762232,0.736225,1.243260,1.008142,...,-0.148252,-0.357958,0.750365,-0.682957,0,1,0,0,0,0
2,1030734575,1.066221,0.560322,0.674694,1.148817,1.485084,1.416890,0.682625,1.441734,0.749672,...,-1.389552,-0.172274,-2.000654,0.211343,0,0,0,0,0,0


In [42]:
print(f"Data types (first 5 columns):\n{data.dtypes.head(5)}")
print(f"\nTotal missing values: {data.isnull().sum().sum()}")
data.describe().iloc[:, :5]

Data types (first 5 columns):
start         int64
mis_link    float64
silence     float64
code_dev    float64
file        float64
dtype: object

Total missing values: 0


,start,mis_link,silence,code_dev,file
count,4.870000e+03,4870.000000,4870.000000,4870.000000,4870.000000
mean,1.192686e+09,0.013419,0.007500,0.008059,0.000824
std,1.374231e+08,1.000000,1.007336,1.006393,1.030376
min,9.142376e+08,-1.644976,-1.803194,-2.511340,-3.662150
25%,1.092687e+09,-0.697270,-0.691262,-0.834315,-0.688600
50%,1.194174e+09,-0.025338,-0.028273,0.200191,0.003467
75%,1.303038e+09,0.794371,0.700618,0.674694,0.688267
max,1.497438e+09,3.662150,2.511340,3.549522,3.662150


In [43]:
# SEM-BIC requires continuous numerical data
data = data.astype({col: "float64" for col in data.columns})

---

## 3. Configure Score Parameters (SEM-BIC)

**Task 1 of 3: Score Configuration**

The **SEM-BIC score** evaluates how well a potential causal graph fits the data. It balances:
- **Fit**: How well the model explains the data
- **Complexity**: How many edges/parameters the model has

### R Code (from Kumu):
```r
score_flags <- score_sem_bic(penalty_discount = 2, 
                             sem_bic_rule = 1, 
                             sem_bic_structure_prior = 0, 
                             precompute_covariances = TRUE)
```

### Python Equivalent:
Looking at [TetradSearch.py](https://github.com/cmu-phil/py-tetrad/blob/main/pytetrad/tools/TetradSearch.py#L79-L84):
```python
def use_sem_bic(self, penalty_discount=2, structurePrior=0, sem_bic_rule=1, singularity_lambda=0.0):
    self.params.set(Params.PENALTY_DISCOUNT, penalty_discount)
    self.params.set(Params.SEM_BIC_STRUCTURE_PRIOR, structurePrior)
    self.params.set(Params.SEM_BIC_RULE, sem_bic_rule)
    self.params.set(Params.SINGULARITY_LAMBDA, singularity_lambda)
    self.SCORE = score_.SemBicScore()
```

### Parameter Mapping:

| R Parameter | Python Parameter | Value | Description |
|-------------|------------------|-------|-------------|
| `penalty_discount` | `penalty_discount` | `2` | Controls the penalty for model complexity. Higher = prefer simpler models |
| `sem_bic_rule` | `sem_bic_rule` | `1` | Lambda calculation method: 1=Chickering, 2=Nandy |
| `sem_bic_structure_prior` | `structurePrior` | `0` | Prior belief about graph structure (0 = no prior) |
| `precompute_covariances` | N/A | `TRUE` | **Not needed in Python** - handled internally by Java |

**Note on `precompute_covariances`**: In Kumu's R code, this tells Tetrad-CMD whether to precompute covariance matrices. In PyTetrad, this is handled automatically by the Java backend, so we don't need to set it explicitly.

In [44]:
# Initialize TetradSearch with our data
search = ts.TetradSearch(data)

# Configure SEM-BIC score (matching R parameters)
search.use_sem_bic(
    penalty_discount=2,          # R: penalty_discount = 2
    sem_bic_rule=1,              # R: sem_bic_rule = 1 (Chickering)
    structurePrior=0,            # R: sem_bic_structure_prior = 0
    singularity_lambda=0.0
)

---

## 4. Configure Algorithm & Bootstrapping Parameters

**Task 2 of 3: Combined Algorithm & Bootstrapping Configuration**

This section configures both the BOSS algorithm and bootstrapping together, since they're closely related (e.g., BOSS uses 100% resample size due to its internal randomization).

### 4.1 Bootstrapping Configuration

**Bootstrapping** makes causal discovery more robust by:
1. Creating multiple **resampled versions** of the dataset
2. Running BOSS on each resample
3. Combining results to find **stable causal relationships**

### R Code (from Kumu) - BOSS Version:
```r
bootstrapping_flags <- bootstrapping(
    number_resampling=1000,
    percent_resample_size = 100,
    seed = 32, 
    add_original_dataset = TRUE,
    resampling_with_replacement = TRUE,
    resampling_ensemble = 1,
    save_bootstrap_graphs = FALSE

)**Note**: BOSS uses 100% resample size (not 90%) because the algorithm already has random initialization, unlike FGES which benefits from smaller resamples (500 iterations at 90%). See commented code in bootstrap configuration cell for FGES parameters.

```

| R Parameter | Python Parameter | Value | Description |
|-------------|------------------|-------|-------------|
| `number_resampling` | `numberResampling` | `1000` | Number of bootstrap iterations |
| `percent_resample_size` | `percent_resample_size` | `100` | Each resample uses 100% of the data (BOSS-specific) |
| `seed` | `seed` | `32` | Random seed for reproducibility |
| `add_original_dataset` | `add_original` | `TRUE` | Include full dataset alongside resamples |
| `resampling_with_replacement` | `with_replacement` | `TRUE` | Bootstrap sampling (with replacement) |
| `resampling_ensemble` | `resampling_ensemble` | `1` | Ensemble method: 1=Preserved, 2=Highest, 3=Majority |
| `save_bootstrap_graphs` | N/A | `FALSE` | **Not in Python API** - graphs stored in memory |

**Note on `save_bootstrap_graphs`**: The Python API stores bootstrap graphs in memory (accessible via `search.bootstrap_graphs`) rather than saving to disk, so this parameter isn't needed.

The **BOSS algorithm** controls how the causal search is performed using permutation-based approaches. These parameters are configured together with bootstrapping since BOSS's internal randomization affects the bootstrap configuration (100% resample size instead of 90%).

---

### 4.2 Algorithm Configuration (BOSS)

In [ ]:
# Configure bootstrapping (matching R parameters for BOSS)
# BOSS uses 100% resample size because the algorithm has random initialization
search.set_bootstrapping(
    numberResampling=1000,        # R: number_resampling=1000
    percent_resample_size=100,    # R: percent_resample_size=100 (BOSS uses 100%)
    seed=32,                      # R: seed=32
    add_original=True,            # R: add_original_dataset=TRUE
    with_replacement=True,        # R: resampling_with_replacement=TRUE
    resampling_ensemble=1         # R: resampling_ensemble=1 (Preserved)
)

# ============================================================
# Alternative Bootstrapping for FGES (Commented Out)
# ============================================================
# search.set_bootstrapping(
#     numberResampling=1000,         # FGES: 1000 iterations
#     percent_resample_size=90,     # FGES: 90% resample size (not 100%)
#     seed=32,
#     add_original=True,
#     with_replacement=True,
#     resampling_ensemble=1
# )

search.set_verbose(verbose=False)  # R: verbose

# Set additional BOSS parameterssearch.set_time_lag(time_lag=0)    # R: time_lag


### 4.3 BOSS Algorithm Parameters

The **BOSS algorithm** parameters control how the causal search is performed using permutation-based approaches.

#### R Code (from Kumu):
```r
algorithm_flags <- algorithm_boss(
    num_starts = 1,
    num_threads = 15,
    time_lag = 0,
    use_bes = FALSE,
    use_data_order = TRUE,
    verbose = FALSE
)
```

#### Python Equivalent (PyTetrad):

| R Parameter | Python Parameter | Value | Description | Available? |
|-------------|------------------|-------|-------------|------------|
| `num_starts` | `num_starts` | `1` | Number of random restarts | ✓ **Direct parameter** |
| `time_lag` | `time_lag` | `0` | For time-series data (0=no time lag) | ✓ **Direct parameter** |
| `use_bes` | `use_bes` | `FALSE` | Use BES (Backward Equivalence Search) from GES | ✓ **Direct parameter** |
| `use_data_order` | `use_data_order` | `TRUE` | Use data variable order for first permutation | ✓ **Direct parameter** |
| `verbose` | Via `set_verbose()` | `FALSE` | Print detailed output | ✓ **Separate method** |
| `num_threads` | Via Java API | `15` | **Not in Python wrapper** | ✗ **Need Java API** |

### Alternative: FGES Algorithm (Commented Out)

For reference, here's how the **FGES algorithm** parameters work:

#### R Code (from Kumu):
```r
algorithm_flags <- algorithm_fges(
---
    time_lag = 0,
### 4.4 Alternative: FGES Algorithm & Bootstrapping (Commented Out)
    meek_verbose = FALSE,
For reference, here's how the **FGES algorithm** parameters and bootstrapping work together:

#### R Code (from Kumu):
```r
algorithm_flags <- algorithm_fges(
    max_degree = 1000,
    time_lag = 0,

#### FGES Python Parameters (For Reference):

| R Parameter | Python Parameter | Value | Description | Available? |
|-------------|------------------|-------|-------------|------------|
| `max_degree` | `max_degree` | `1000` | Maximum number of edges per node | ✓ **Direct parameter** |
| `time_lag` | Via `set_time_lag()` | `0` | For time-series data (0=no time lag) | ✓ **Separate method** |
| `faithfulness_assumed` | `faithfulness_assumed` | `TRUE` | Assume faithfulness condition | ✓ **Direct parameter** |
| `symmetric_first_step` | `symmetric_first_step` | `TRUE` | Use symmetric scoring in first step | ✓ **Direct parameter** |
| `parallelized` | `parallelized` | `FALSE` | Use parallel processing | ✓ **Direct parameter** |
| `verbose` | Via `set_verbose()` | `TRUE` | Print detailed output | ✓ **Separate method** |
| `meek_verbose` | Via Java API | `FALSE` | **Not in Python wrapper** | ✗ **Need Java API** |

**Note**: The FGES implementation is included as commented code for reference throughout this notebook.

---

## 5. Load Knowledge Constraints

**Knowledge constraints** allow you to incorporate prior domain knowledge to guide the causal discovery process. You can specify:
- **Temporal tiers**: Variables in earlier tiers cannot be caused by variables in later tiers
- **Forbidden edges**: Specific causal relationships that should not appear in the graph
- **Required edges**: Specific causal relationships that must appear in the graph

### How to Use Different Knowledge Files (Matching Kumu's Design)

Just like in Kumu's R implementation where `knowledge_flags` is optional, you can easily interchange knowledge files:

**In Kumu (R):**
```r
# With knowledge
result <- tetrad(..., knowledge_flags = knowledge_file_path("my_knowledge.txt"), ...)

# Without knowledge (default)
result <- tetrad(..., knowledge_flags = list(), ...)
```

**In PyTetrad (Python) - This Notebook:**
```python
# Simply change this one line at the top of the cell:
knowledge_file = "mike_knowledge_box.txt"    # Use this knowledge file
knowledge_file = "other_knowledge.txt"       # Or use a different file
knowledge_file = None                        # Or run without knowledge
```

The code below handles all three cases automatically, matching Kumu's pattern where `if(length(knowledge_flags) == 0)` checks for empty knowledge.

### Python Implementation Details

Looking at [TetradSearch.py](https://github.com/cmu-phil/py-tetrad/blob/main/pytetrad/tools/TetradSearch.py#L312-L335):
```python
def load_knowledge(self, path, use_for_mc=False):
    know_file = io.File(path)
    know_delim = td.DelimiterType.WHITESPACE
    self.knowledge = td.SimpleDataLoader.loadKnowledge(know_file, know_delim, "#")
```

The `run_boss()` method automatically applies the knowledge via `alg.setKnowledge(self.knowledge)`.

### Knowledge File Format

Knowledge files use a simple text format:
```
/knowledge
addtemporal

0 var1 var2 var3    # Tier 0: earliest variables
1 var4 var5         # Tier 1: variables that can be caused by tier 0

forbiddirect
var1 var2           # Forbid edge var1 -> var2

requiredirect
var3 var4           # Require edge var3 -> var4
```

### Knowledge File Format

Knowledge files use a simple text format compatible with Tetrad-CMD:
```
/knowledge
addtemporal

0 var1 var2 var3    # Tier 0: earliest variables (cannot be caused by later tiers)
1 var4 var5         # Tier 1: can be caused by tier 0, but not by tier 2

forbiddirect
var1 var2           # Forbid edge var1 -> var2

requiredirect
var3 var4           # Require edge var3 -> var4
```

### Examples of Different Configurations

```python
# Example 1: Use mike_knowledge_box.txt (current)
knowledge_file = "mike_knowledge_box.txt"

# Example 2: Use a different knowledge file
knowledge_file = "domain_expert_knowledge.txt"

# Example 3: Use full path to knowledge file
knowledge_file = "/path/to/my_project/knowledge_constraints.txt"

# Example 4: No knowledge constraints (run unconstrained search)
knowledge_file = None
# or
knowledge_file = ""
```

In [47]:
# Knowledge File (Optional) - set to None to run without constraints
knowledge_file = "mike_knowledge_box.txt"

if knowledge_file and os.path.exists(knowledge_file):
    search.load_knowledge(knowledge_file)
    
    # Validate knowledge variables against dataset
    knowledge_var_names = [str(var) for var in search.knowledge.getVariables()]
    matched = [v for v in knowledge_var_names if v in set(data.columns)]
    print(f"Knowledge loaded: {len(matched)}/{len(knowledge_var_names)} variables matched dataset")
elif knowledge_file:
    print(f"⚠️  Knowledge file not found: {knowledge_file}")
else:
    print("Running without knowledge constraints")


Loading knowledge.
Adding to tier 0 anti_motif_square
Adding to tier 0 anti_triangle_motif
Adding to tier 0 clique
Adding to tier 0 crossing
Adding to tier 0 modularity-violation
Adding to tier 0 package-cycle
Adding to tier 0 unhealthy-inheritance
Adding to tier 0 unstable-interface
Adding to tier 1 file_bug_churn
Adding to tier 1 file_bug_frequency
Adding to tier 1 file_churn
Adding to tier 1 file_non_bug_churn
Adding to tier 1 file_non_bug_frequency
Knowledge loaded: 0/13 variables matched dataset


---

## 6. Run BOSS/FGES Search with All Configurations

### 6.1 Run FGES Search


Now we're ready to execute the FGES algorithm with all our configurations, including the knowledge constraints we loaded in Section 5.5. The knowledge will automatically be applied during the search via `alg.setKnowledge(self.knowledge)` inside `run_fges()`.

In [48]:
# FGES alternative (uncomment to use instead of BOSS)
# Requires reconfiguring bootstrapping: 500 iterations, 90% resample size

# import time
# start_time = time.time()
# search.run_fges(
#     max_degree=1000,
#     faithfulness_assumed=True,
#     symmetric_first_step=True,
#     parallelized=False
# )
# elapsed_time = time.time() - start_time
# print(f"FGES completed in {elapsed_time:.0f}s ({elapsed_time/60:.1f} min)")

### 6.2 Run BOSS Search

Now we're ready to execute the BOSS algorithm with all our configurations, including the knowledge constraints we loaded. The knowledge will automatically be applied during the search via `alg.setKnowledge(self.knowledge)` inside `run_boss()`.

In [49]:
import time

start_time = time.time()

# Suppress Java output during bootstrap iterations
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    search.run_boss(
        num_starts=1,              # R: num_starts = 1
        use_bes=False,             # R: use_bes = FALSE
        time_lag=0,                # R: time_lag = 0
        use_data_order=True,       # R: use_data_order = TRUE
        output_cpdag=True          # Output CPDAG format
    )

elapsed_time = time.time() - start_time
print(f"BOSS completed in {elapsed_time:.0f}s ({elapsed_time/60:.1f} min)")

Bootstrap count = 1
Bootstrap count = 2
Bootstrap count = 3
Bootstrap count = 4
Bootstrap count = 5
Bootstrap count = 6
Bootstrap count = 7
Bootstrap count = 8
Bootstrap count = 9
Bootstrap count = 10
Bootstrap count = 11
Bootstrap count = 12
Bootstrap count = 13
Bootstrap count = 14
Bootstrap count = 15
Bootstrap count = 16
Bootstrap count = 17
Bootstrap count = 18
Bootstrap count = 19
Bootstrap count = 20
Bootstrap count = 21
Bootstrap count = 22
Bootstrap count = 23
Bootstrap count = 24
Bootstrap count = 25
Bootstrap count = 26
Bootstrap count = 27
Bootstrap count = 28
Bootstrap count = 29
Bootstrap count = 30
Bootstrap count = 31
Bootstrap count = 32
Bootstrap count = 33
Bootstrap count = 34
Bootstrap count = 35
Bootstrap count = 36
Bootstrap count = 37
Bootstrap count = 38
Bootstrap count = 39
Bootstrap count = 40
Bootstrap count = 41
Bootstrap count = 42
Bootstrap count = 43
Bootstrap count = 44
Bootstrap count = 45
Bootstrap count = 46
Bootstrap count = 47
Bootstrap count = 48
B

### What Just Happened?

1. **BOSS ran 1001 times** (1 original + 1000 bootstraps)
2. Each run used the configured parameters (100% resample size, seed=32, etc.)
3. Each run discovered potential causal relationships using permutation-based approaches
4. Results were combined using the "Preserved" ensemble method
5. The final graph represents **stable causal relationships** that appeared consistently across runs
6. **Knowledge constraints** were applied to guide the search based on domain expertise

The result is stored in `search.java` (the Java graph object) and `search.bootstrap_graphs` (all individual bootstrap results).

---

## 7. Visualize and Export Results

Let's examine the causal graph discovered by BOSS.

In [50]:
graph = search.get_java()
print(f"Nodes: {graph.getNumNodes()}, Edges: {graph.getNumEdges()}")

# Graph string (preview first 500 chars)
graph_string = search.get_string()
print(f"\nGraph preview:\n{str(graph_string)[:500]}\n...")

Nodes: 138, Edges: 95

Graph preview:
Graph Nodes:
b_100433;b_100740;b_100742;b_102939;b_103864;b_104180;b_113207;b_114109;b_114576;b_114577;b_114619;b_120027;b_120884;b_122110;b_122333;b_130166;b_134353;b_136450;b_140076;b_140160;b_140195;b_140221;b_140224;b_142970;b_143470;b_143505;b_143506;b_143507;b_143508;b_143509;b_143510;b_143511;b_143513;b_143567;b_143568;b_143569;b_143570;b_143571;b_143572;b_148275;b_150204;b_150205;b_150206;b_150207;b_150208;b_150209;b_150285;b_150286;b_150287;b_150288;b_150289;b_150290;b_150291;b_151787;b
...


In [51]:
# Adjacency matrix (encoding: 0=none, 1=circle, 2=arrow, 3=tail)
adjacency_matrix = search.get_graph_to_matrix()

print(f"Adjacency Matrix: {adjacency_matrix.shape}")
adjacency_matrix.iloc[:5, :5]

Adjacency Matrix: (138, 138)


,b_100433,b_100740,b_100742,b_102939,b_103864
0,0,0,0,0,0
1,0,0,0,0,0
2,0,0,0,0,0
3,0,0,0,0,0
4,0,0,0,0,0


In [52]:
# Extract edges from the adjacency matrix
edges = []
for i, var1 in enumerate(adjacency_matrix.columns):
    for j, var2 in enumerate(adjacency_matrix.columns):
        if i < j:
            endpoint1 = adjacency_matrix.iloc[i, j]
            endpoint2 = adjacency_matrix.iloc[j, i]
            if endpoint1 != 0 or endpoint2 != 0:
                edges.append({
                    'From': var1, 'To': var2,
                    'Endpoint_From': endpoint1, 'Endpoint_To': endpoint2
                })

edges_df = pd.DataFrame(edges)
print(f"Total edges: {len(edges_df)}")
edges_df.head(5)

Total edges: 95


,From,To,Endpoint_From,Endpoint_To
0,b_130166,nv-b_104180,2,3
1,b_160705,start,3,2
2,b_162107,start,3,2
3,b_162180,start,3,2
4,b_173733,nv-b_102939,2,3


In [53]:
# Save results
output_dir = "boss_results"
os.makedirs(output_dir, exist_ok=True)

adjacency_matrix.to_csv(f"{output_dir}/adjacency_matrix.csv")
edges_df.to_csv(f"{output_dir}/edge_list.csv", index=False)
with open(f"{output_dir}/graph_string.txt", "w") as f:
    f.write(str(graph_string))

print(f"Results saved to {output_dir}/")
print(f"  adjacency_matrix.csv, edge_list.csv, graph_string.txt")

Results saved to boss_results/
  adjacency_matrix.csv, edge_list.csv, graph_string.txt


### Understanding the Graph Output

The BOSS algorithm produces a **CPDAG (Completed Partially Directed Acyclic Graph)**, which means:

1. **Directed edges (→)**: Causal direction is determined (X causes Y)
2. **Undirected edges (—)**: Causal direction cannot be determined (X and Y are related, but direction is ambiguous)

**Edge Types:**
- `X → Y` (tail-arrow): X causes Y
- `X — Y` (tail-tail): X and Y are related, but direction unknown
- `X ↔ Y` (arrow-arrow): Not typically in BOSS output, but would indicate selection bias

**Adjacency Matrix Encoding:**
- `adjacency_matrix[i,j] = 2` and `adjacency_matrix[j,i] = 3` means: i → j (i causes j)
- `adjacency_matrix[i,j] = 3` and `adjacency_matrix[j,i] = 3` means: i — j (undirected)

---

## Summary: R to Python Migration

### What We Accomplished

We successfully migrated the key components from Kumu's R implementation to Python, with both BOSS and FGES algorithms:

| Component | R (Kumu) | Python (PyTetrad) | Status |
|-----------|----------|-------------------|--------|
| **1. Score (SEM-BIC)** | `score_sem_bic()` | `search.use_sem_bic()` | ✅ Complete |
| **2. Bootstrapping** | `bootstrapping()` | `search.set_bootstrapping()` | ✅ Complete (both BOSS & FGES) |
| **3. BOSS Algorithm** | `algorithm_boss()` | `search.run_boss()` | ✅ Complete (active) |
| **3. FGES Algorithm** | `algorithm_fges()` | `search.run_fges()` | ✅ Complete (commented) |
| **4. Knowledge Constraints** | `knowledge_file_path()` | `search.load_knowledge()` | ✅ Complete |

### Algorithm-Specific Parameters

**BOSS (Active)**:
- 1000 bootstrap iterations
- 100% resample size (due to internal randomization)
- Knowledge constraints supported

**FGES (Commented)**:
- 500 bootstrap iterations
- 90% resample size
- Knowledge constraints supported
- See commented code blocks throughout notebook

### Key Differences

1. **Execution Model:**
   - **R (Kumu)**: Constructs command-line flags → Calls Tetrad-CMD via `system2()` → Writes output to disk
   - **Python (PyTetrad)**: Configures Java objects in-memory → Runs directly via JPype → Returns Java objects

## Complete Parameter Reference

For your reference, here's a complete mapping of all parameters from the R code to Python for both algorithms:

### Score Parameters (SEM-BIC) - Same for Both Algorithms
```python
search.use_sem_bic(
    penalty_discount=2,      # R: penalty_discount
    sem_bic_rule=1,          # R: sem_bic_rule (1=Chickering, 2=Nandy)
    structurePrior=0,        # R: sem_bic_structure_prior
    singularity_lambda=0.0   # Not in R code (Python default)
)
# R parameter 'precompute_covariances' not needed - handled by Java
```

### Bootstrapping Parameters - BOSS (Active)
```python
search.set_bootstrapping(
    numberResampling=1000,        # R: number_resampling (1000 for BOSS)
    percent_resample_size=100,    # R: percent_resample_size (100% for BOSS)
    seed=32,                      # R: seed
    add_original=True,            # R: add_original_dataset
    with_replacement=True,        # R: resampling_with_replacement
    resampling_ensemble=1         # R: resampling_ensemble (1=Preserved)
)
# Note: BOSS uses 100% resample size due to internal randomization
```

### Bootstrapping Parameters - FGES (Commented Alternative)
```python
# search.set_bootstrapping(
#     numberResampling=500,         # R: number_resampling (500 for FGES)
#     percent_resample_size=90,     # R: percent_resample_size (90% for FGES)
#     seed=32,                      # R: seed
#     add_original=True,            # R: add_original_dataset
#     with_replacement=True,        # R: resampling_with_replacement
#     resampling_ensemble=1         # R: resampling_ensemble (1=Preserved)
# )
```

### BOSS Algorithm Parameters (Active)
```python
search.set_time_lag(time_lag=0)    # R: time_lag
search.set_verbose(verbose=False)  # R: verbose

search.run_boss(
    num_starts=1,              # R: num_starts
    use_bes=False,             # R: use_bes
    time_lag=0,                # R: time_lag (can also set via set_time_lag)
    use_data_order=True,       # R: use_data_order
    output_cpdag=True          # Output CPDAG format
)
# R parameter 'num_threads' not in Python wrapper - handled by Java
```

### FGES Algorithm Parameters (Commented Alternative)
```python
# search.set_time_lag(time_lag=0)   # R: time_lag
# search.set_verbose(verbose=True)  # R: verbose
# 
# search.run_fges(
#     max_degree=1000,              # R: max_degree
#     faithfulness_assumed=True,    # R: faithfulness_assumed
#     symmetric_first_step=True,    # R: symmetric_first_step
#     parallelized=False            # R: parallelized
# )
# R parameter 'meek_verbose' not in Python wrapper
```

### Knowledge Constraints (Optional) - Same for Both Algorithms
```python
search.load_knowledge("knowledge_file.txt")  # R: knowledge_file_path()
# The knowledge is automatically applied during search.run_boss() or search.run_fges()
```

### Complete Workflow - BOSS (Active)
```python
# 1. Initialize with data
search = ts.TetradSearch(data)

# 2. Configure score
search.use_sem_bic(penalty_discount=2, sem_bic_rule=1, structurePrior=0)

# 3. Configure bootstrapping (BOSS parameters)
search.set_bootstrapping(numberResampling=1000, percent_resample_size=100, 
                        seed=32, add_original=True, with_replacement=True, 
                        resampling_ensemble=1)

# 4. Set algorithm parameters
search.set_time_lag(0)
search.set_verbose(False)

# 5. Load knowledge (optional)
search.load_knowledge("knowledge.txt")

# 6. Run BOSS
search.run_boss(num_starts=1, use_bes=False, time_lag=0, 
                use_data_order=True, output_cpdag=True)

# 7. Get results
graph = search.get_java()
adjacency_matrix = search.get_adjacency_matrix()
edges = search.get_edges()
```

### Complete Workflow - FGES (Commented Alternative)
```python
# # 1. Initialize with data
# search = ts.TetradSearch(data)
# 
# # 2. Configure score
# search.use_sem_bic(penalty_discount=2, sem_bic_rule=1, structurePrior=0)
# 
# # 3. Configure bootstrapping (FGES parameters)
# search.set_bootstrapping(numberResampling=500, percent_resample_size=90, 
#                         seed=32, add_original=True, with_replacement=True, 
#                         resampling_ensemble=1)
# 
# # 4. Set algorithm parameters
# search.set_time_lag(0)
# search.set_verbose(True)
# 
# # 5. Load knowledge (optional)
# search.load_knowledge("knowledge.txt")
# 
# # 6. Run FGES
# search.run_fges(max_degree=1000, faithfulness_assumed=True,
#                 symmetric_first_step=True, parallelized=False)
# 
# # 7. Get results
# graph = search.get_java()
# adjacency_matrix = search.get_adjacency_matrix()
# edges = search.get_edges()
```